# BISINDO WL15 - fold s4 on Google Colab (6 runs)

Decided 31 Oct 2026 (config_log section 28): fold s4 (plan `colab_s4`) runs here; P1 and folds s1-s3 run on Kaggle.
Same dataset zip as Kaggle dataset version of 31 Oct 2026 (checked by sha256), same session script
(`scripts/kaggle_session.py` with `BISINDO_PLATFORM=colab`): pinned packages, albumentations removed, the same
guards, stop if the GPU is not a Tesla T4, batch 16, checkpoints every epoch.

**Before the first run**
1. Upload `kaggle_upload/bisindo-wl15.zip` (laptop) to Google Drive: `MyDrive/bisindo-colab/bisindo-wl15.zip`.
2. Runtime > Change runtime type > **T4 GPU**.

**Run:** Runtime > Run all. Keep this tab open and the laptop awake until the last cell has finished.

**If the runtime disconnects or stops:** reconnect (T4 GPU) and Run all again. `runs/` is on Google Drive, so
finished runs are skipped and the unfinished one resumes from its last saved epoch (`last.pt`). The log line
"restore: no earlier session attached" is expected when the first session stopped before writing its report.

**At the end:** the last cell writes `MyDrive/bisindo-colab/export/bisindo_colab_runs_<time>.zip` (everything
except `last.pt`). Download it to the laptop and import it with
`python scripts/kaggle_download.py --platform colab --zip <zip>`.

In [ ]:
import os, subprocess, sys

DRIVE_DIR = "/content/drive/MyDrive/bisindo-colab"   # holds bisindo-wl15.zip; runs/ and export/ are written here
ZIP_SHA256 = "6d7d7dedaebdc9d02b326376702a346d831e7c081c4ec347a02b025af398a77f"   # = Kaggle dataset version of 31 Oct 2026
PLAN = "colab_s4"
# Session length limit in hours, counted from the start of the session script (at most 23.5). No epoch is started
# that would end after it. Set it below what the remaining compute units allow: balance / (units per hour) - 0.5.
MAX_HOURS = 23.0

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu, "| CPUs:", os.cpu_count(), "| Python:", sys.version.split()[0])
assert gpu.split(",")[0].strip() == "Tesla T4", (
    f"GPU is {gpu!r}, not a Tesla T4: Runtime > Change runtime type > T4 GPU, then Run all again")
# Ultralytics uses min(CPUs, workers=2) data-loader workers; with fewer than 2 CPUs the data order would differ
assert os.cpu_count() >= 2, "fewer than 2 CPUs"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import hashlib, shutil, zipfile
from pathlib import Path

src = Path(DRIVE_DIR) / "bisindo-wl15.zip"
assert src.exists(), f"upload bisindo-wl15.zip to {DRIVE_DIR} first"
local_zip = Path("/content/bisindo-wl15.zip")
if not local_zip.exists():
    shutil.copy(src, local_zip)
h = hashlib.sha256()
with open(local_zip, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == ZIP_SHA256, f"zip sha256 {h.hexdigest()} != {ZIP_SHA256}: not the 31 Oct 2026 dataset zip"
print("dataset zip OK (same as the Kaggle dataset version)")

DATASET = Path("/content/bisindo-wl15")          # local copy: fast reads during training
if not (DATASET / "manifest_sha256.txt").exists():
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(DATASET)
WORK = Path("/content/bisindo")
WORK.mkdir(exist_ok=True)
runs_drive = Path(DRIVE_DIR) / "runs"            # checkpoints and logs: on Drive, kept between sessions
runs_drive.mkdir(parents=True, exist_ok=True)
if not (WORK / "runs").is_symlink():
    (WORK / "runs").symlink_to(runs_drive, target_is_directory=True)
print("runs/ ->", os.path.realpath(WORK / "runs"), "| already there:",
      sorted(p.name for p in runs_drive.glob("*/*") if p.is_dir()))


In [ ]:
# Training. The child's output is printed line by line (epoch summaries, not progress bars).
env = {**os.environ, "BISINDO_PLATFORM": "colab", "BISINDO_DATASET": str(DATASET), "BISINDO_WORK": str(WORK),
       "BISINDO_INPUT": "/content/no_input"}
proc = subprocess.Popen([sys.executable, "-u", str(DATASET / "scripts" / "kaggle_session.py"), "--plan", PLAN,
                         "--max-hours", str(MAX_HOURS)], env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
print("exit code:", proc.wait())


In [ ]:
# Export for the laptop: runs/ without last.pt (best.pt, eval_test.json, predictions, CSV logs, reports).
import time
export = Path(DRIVE_DIR) / "export"
export.mkdir(exist_ok=True)
out = export / f"bisindo_colab_runs_{time.strftime('%Y%m%d-%H%M', time.gmtime())}.zip"
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(runs_drive.rglob("*")):
        if p.is_file() and p.name != "last.pt":
            z.write(p, "bisindo/runs/" + p.relative_to(runs_drive).as_posix())
            n += 1
print(f"{n} files -> {out} ({out.stat().st_size / 2**20:.0f} MB)")
for r in sorted(runs_drive.glob("*/*/eval_test.json")):
    print("finished:", r.parent.name)
drive.flush_and_unmount()   # make sure everything is written to Drive
